In [ ]:
from pathlib import Path
import sqlite3
import pandas as pd

root = Path.cwd()
while not (root / "data" / "db").exists() and root != root.parent:
    root = root.parent

con = sqlite3.connect(root / "data" / "db" / "tennis.db")
games = pd.read_sql("""
    SELECT match_id, tourney_start, file_year, winner_id, loser_id
    FROM matches
    ORDER BY match_id
""", con)

print(games.shape)
print(games.isna().sum().to_string())

In [ ]:
from pathlib import Path
import sqlite3
import numpy as np
import pandas as pd

root = Path.cwd()
while not (root / "data" / "db").exists() and root != root.parent:
    root = root.parent

con = sqlite3.connect(root / "data" / "db" / "tennis.db")
games = pd.read_sql("""
    SELECT match_id, file_year, tourney_start, tourney_level,
           winner_id, loser_id, winner_rank, loser_rank
    FROM matches
    ORDER BY match_id
""", con)
games["day"] = (pd.to_datetime(games["tourney_start"]) - pd.Timestamp("2000-01-01")).dt.days

print(games.shape)
print(list(games.columns))
print(games.isna().sum().to_string())
print(games["tourney_level"].value_counts().to_string())

In [ ]:
def score(res, y0, y1):
    d = res[(res["file_year"] >= y0) & (res["file_year"] <= y1)]
    p = d["exp_w"]
    return {
        "years": f"{y0}-{y1}",
        "matches": len(d),
        "accuracy_%": round(100 * ((p > 0.5).mean() + 0.5 * (p == 0.5).mean()), 1),
        "log_loss": round(float(-np.log(p).mean()), 4),
        "brier": round(float(((1 - p) ** 2).mean()), 4),
    }

res, final_rating, final_played = run_elo(games)
print(pd.DataFrame([score(res, 2005, 2018), score(res, 2019, 2026)]).to_string(index=False))

g = games[games["winner_rank"].notna() & games["loser_rank"].notna() & (games["winner_rank"] != games["loser_rank"])]
for y0, y1 in [(2005, 2018), (2019, 2026)]:
    d = g[(g["file_year"] >= y0) & (g["file_year"] <= y1)]
    print(f"ATP ranking baseline {y0}-{y1}: {round(100 * (d['winner_rank'] < d['loser_rank']).mean(), 1)}%")

In [ ]:
test, _, _ = run_elo(games.head(1000))
print(test.shape)
print(test.head(3)[["match_id", "r_winner", "r_loser", "exp_w"]].round(3).to_string(index=False))

In [ ]:
res, final_rating, final_played = run_elo(games)
print(res.shape)

In [ ]:
def score(res, y0, y1):
    d = res[(res["file_year"] >= y0) & (res["file_year"] <= y1)]
    p = d["exp_w"]
    return {
        "years": f"{y0}-{y1}",
        "matches": len(d),
        "accuracy_%": round(100 * ((p > 0.5).mean() + 0.5 * (p == 0.5).mean()), 1),
        "log_loss": round(float(-np.log(p).mean()), 4),
        "brier": round(float(((1 - p) ** 2).mean()), 4),
    }

print(pd.DataFrame([score(res, 2005, 2018), score(res, 2019, 2026)]).to_string(index=False))

g = games[games["winner_rank"].notna() & games["loser_rank"].notna() & (games["winner_rank"] != games["loser_rank"])]
for y0, y1 in [(2005, 2018), (2019, 2026)]:
    d = g[(g["file_year"] >= y0) & (g["file_year"] <= y1)]
    print(f"ATP ranking baseline {y0}-{y1}: {round(100 * (d['winner_rank'] < d['loser_rank']).mean(), 1)}%")

In [ ]:
m = res.merge(games[["match_id", "winner_rank", "loser_rank"]], on="match_id")
m = m[m["winner_rank"].notna() & m["loser_rank"].notna()
      & (m["winner_rank"] != m["loser_rank"]) & (m["exp_w"] != 0.5)].copy()
m["elo_ok"] = (m["exp_w"] > 0.5).astype(int)
m["rank_ok"] = (m["winner_rank"] < m["loser_rank"]).astype(int)

for y0, y1 in [(2005, 2018), (2019, 2026)]:
    d = m[(m["file_year"] >= y0) & (m["file_year"] <= y1)]
    diff = d["elo_ok"] - d["rank_ok"]
    se = diff.std() / np.sqrt(len(d))
    print(f"{y0}-{y1}: n={len(d)}  Elo {100*d['elo_ok'].mean():.1f}%  ATP rank {100*d['rank_ok'].mean():.1f}%  "
          f"difference {100*diff.mean():.2f} points (95% interval {100*(diff.mean()-1.96*se):.2f} to {100*(diff.mean()+1.96*se):.2f})")

In [ ]:
d = res[(res["file_year"] >= 2005) & (res["file_year"] <= 2018) & (res["exp_w"] != 0.5)].copy()
d["p_fav"] = np.maximum(d["exp_w"], 1 - d["exp_w"])
d["fav_won"] = (d["exp_w"] > 0.5).astype(int)
d["bin"] = pd.cut(d["p_fav"], [0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 1.0])

t = d.groupby("bin", observed=True).agg(
    matches=("fav_won", "size"),
    predicted_pct=("p_fav", lambda s: round(100 * s.mean(), 1)),
    actual_pct=("fav_won", lambda s: round(100 * s.mean(), 1)),
)
print(t.to_string())

In [ ]:
rows = []
for k_old in [10, 15, 20, 25, 30]:
    for k_new in [20, 30, 40]:
        r_, _, _ = run_elo(games, k_new=k_new, k_old=k_old)
        s = score(r_, 2005, 2018)
        rows.append({"k_old": k_old, "k_new": k_new, "accuracy_%": s["accuracy_%"], "log_loss": s["log_loss"], "brier": s["brier"]})

grid = pd.DataFrame(rows).sort_values("log_loss")
print(grid.to_string(index=False))

In [ ]:
#weight by tournament
variants = {
    "baseline (no weights)": {},
    "slams x1.25": {"G": 1.25},
    "slams 1.25, masters/finals 1.1": {"G": 1.25, "M": 1.1, "F": 1.1},
    "slams 1.5, masters/finals 1.25": {"G": 1.5, "M": 1.25, "F": 1.25},
    "250s x0.8": {"250": 0.8},
    "gradient": {"G": 1.3, "M": 1.15, "F": 1.15, "500": 1.0, "250": 0.85},
}

rows = []
for name, mult in variants.items():
    r_, _, _ = run_elo(games, level_mult=mult)
    s = score(r_, 2005, 2018)
    rows.append({"variant": name, "accuracy_%": s["accuracy_%"], "log_loss": s["log_loss"], "brier": s["brier"]})

print(pd.DataFrame(rows).sort_values("log_loss").to_string(index=False))

In [ ]:
variants2 = {
    "baseline": {},
    "gap 180d, K x1.5": dict(gap_days=180, gap_mult=1.5),
    "gap 365d, K x1.5": dict(gap_days=365, gap_mult=1.5),
    "gap 180d, +5 win bonus": dict(gap_days=180, win_bonus=5.0),
}

rows = []
for name, kw in variants2.items():
    r_, _, _ = run_elo(games, **kw)
    s = score(r_, 2005, 2018)
    rows.append({"variant": name, "accuracy_%": s["accuracy_%"], "log_loss": s["log_loss"], "brier": s["brier"]})

print(pd.DataFrame(rows).sort_values("log_loss").to_string(index=False))

In [ ]:
def run_elo(games, start=1450, k_new=40, k_old=20, n_new=30, level_mult=None,
            gap_days=None, gap_mult=1.0, win_bonus=0.0, scale=400):
    level_mult = level_mult or {}
    rating, played, last, out = {}, {}, {}, []
    for r in games.itertuples(index=False):
        w, l = r.winner_id, r.loser_id
        rw, rl = rating.get(w, start), rating.get(l, start)
        nw, nl = played.get(w, 0), played.get(l, 0)
        gw = r.day - last.get(w, r.day)
        gl = r.day - last.get(l, r.day)
        m = level_mult.get(r.tourney_level, 1.0)

        exp_w = 1 / (1 + 10 ** ((rl - rw) / scale))

        kw = (k_new if nw < n_new else k_old) * (gap_mult if gap_days is not None and gw > gap_days else 1.0)
        kl = (k_new if nl < n_new else k_old) * (gap_mult if gap_days is not None and gl > gap_days else 1.0)
        bonus = win_bonus if gap_days is not None and gw > gap_days else 0.0

        new_w = rw + m * kw * (1 - exp_w) + bonus
        new_l = rl + m * kl * (0 - (1 - exp_w))
        rating[w], rating[l] = new_w, new_l
        played[w], played[l] = nw + 1, nl + 1
        last[w], last[l] = r.day, r.day

        out.append((r.match_id, r.file_year, r.tourney_start, w, l, rw, rl, new_w, new_l, exp_w))
    res = pd.DataFrame(out, columns=["match_id", "file_year", "tourney_start", "winner_id", "loser_id",
                                     "r_winner", "r_loser", "r_winner_after", "r_loser_after", "exp_w"])
    return res, rating, played

In [ ]:
rows = []
for scale in [400, 450, 500, 600, 700]:
    r_, _, _ = run_elo(games, scale=scale)
    s = score(r_, 2005, 2018)
    rows.append({"scale": scale, "accuracy_%": s["accuracy_%"], "log_loss": s["log_loss"], "brier": s["brier"]})

print(pd.DataFrame(rows).sort_values("log_loss").to_string(index=False))

In [ ]:
rows = []
for scale in [300, 350, 400]:
    r_, _, _ = run_elo(games, scale=scale)
    s = score(r_, 2005, 2018)
    rows.append({"scale": scale, "accuracy_%": s["accuracy_%"], "log_loss": s["log_loss"], "brier": s["brier"]})

print(pd.DataFrame(rows).sort_values("log_loss").to_string(index=False))

In [ ]:
def run_elo(games, start=1450, k_new=40, k_old=20, n_new=30, level_mult=None,
            gap_days=None, gap_mult=1.0, win_bonus=0.0, scale=400):
    level_mult = level_mult or {}
    rating, played, last, out = {}, {}, {}, []
    for r in games.itertuples(index=False):
        w, l = r.winner_id, r.loser_id
        rw, rl = rating.get(w, start), rating.get(l, start)
        nw, nl = played.get(w, 0), played.get(l, 0)
        gw = r.day - last.get(w, r.day)
        gl = r.day - last.get(l, r.day)
        m = level_mult.get(r.tourney_level, 1.0)

        exp_w = 1 / (1 + 10 ** ((rl - rw) / scale))

        kw = (k_new if nw < n_new else k_old) * (gap_mult if gap_days is not None and gw > gap_days else 1.0)
        kl = (k_new if nl < n_new else k_old) * (gap_mult if gap_days is not None and gl > gap_days else 1.0)
        bonus = win_bonus if gap_days is not None and gw > gap_days else 0.0

        new_w = rw + m * kw * (1 - exp_w) + bonus
        new_l = rl + m * kl * (0 - (1 - exp_w))
        rating[w], rating[l] = new_w, new_l
        played[w], played[l] = nw + 1, nl + 1
        last[w], last[l] = r.day, r.day

        out.append((r.match_id, r.file_year, r.tourney_start, w, l, rw, rl, new_w, new_l, exp_w))
    res = pd.DataFrame(out, columns=["match_id", "file_year", "tourney_start", "winner_id", "loser_id",
                                     "r_winner", "r_loser", "r_winner_after", "r_loser_after", "exp_w"])
    return res, rating, played

In [ ]:
res, final_rating, final_played = run_elo(games)

a = res[["match_id", "tourney_start", "winner_id", "loser_id", "r_winner", "r_winner_after", "exp_w"]].copy()
a.columns = ["match_id", "tourney_start", "player_id", "opponent_id", "elo_before", "elo_after", "expected"]
a["result"] = 1

b = res[["match_id", "tourney_start", "loser_id", "winner_id", "r_loser", "r_loser_after", "exp_w"]].copy()
b.columns = ["match_id", "tourney_start", "player_id", "opponent_id", "elo_before", "elo_after", "expected"]
b["expected"] = 1 - b["expected"]
b["result"] = 0

elo_long = pd.concat([a, b]).sort_values(["match_id", "result"], ascending=[True, False]).reset_index(drop=True)
print(elo_long.shape)
print(elo_long.head(4).round(1).to_string(index=False))

In [ ]:
n = pd.read_sql("""
    SELECT winner_id AS player_id, winner_name AS name, match_id FROM matches
    UNION ALL
    SELECT loser_id, loser_name, match_id FROM matches
""", con)
players = n.sort_values("match_id").drop_duplicates("player_id", keep="last")[["player_id", "name"]]

elo_long["source"] = "Calculated from TennisMyLife match data"
elo_long["licence"] = "Derived from TML data; non-commercial, see data/SOURCES.md"
elo_long["method"] = "Elo v1: start 1450, K=40 first 30 matches then 20, scale 400, overall rating"
players["source"] = "TennisMyLife match files (names as given)"
players["licence"] = "Derived from TML data; non-commercial, see data/SOURCES.md"

elo_long.to_sql("elo_long", con, if_exists="replace", index=False)
players.to_sql("players", con, if_exists="replace", index=False)
con.execute("CREATE INDEX IF NOT EXISTS idx_elo_player ON elo_long(player_id, match_id)")
con.commit()

print(len(players), "players")
print(pd.read_sql("SELECT COUNT(*) AS rows_total FROM elo_long", con).to_string(index=False))

In [ ]:
#elo function
def run_elo(games, start=1450, k_new=40, k_old=20, n_new=30, level_mult=None,
            gap_days=None, gap_mult=1.0, win_bonus=0.0, scale=400,
            upset_cap=None, reset_days=None, reset_frac=1.0):
    level_mult = level_mult or {}
    rating, played, last, out = {}, {}, {}, []
    for r in games.itertuples(index=False):
        w, l = r.winner_id, r.loser_id
        rw, rl = rating.get(w, start), rating.get(l, start)
        nw, nl = played.get(w, 0), played.get(l, 0)
        gw = r.day - last.get(w, r.day)
        gl = r.day - last.get(l, r.day)

        if reset_days is not None:
            if gw > reset_days:
                rw = start + (1 - reset_frac) * (rw - start)
            if gl > reset_days:
                rl = start + (1 - reset_frac) * (rl - start)

        m = level_mult.get(r.tourney_level, 1.0)
        exp_w = 1 / (1 + 10 ** ((rl - rw) / scale))

        surprise = 1 - exp_w
        if upset_cap is not None:
            surprise = min(surprise, upset_cap)

        kw = (k_new if nw < n_new else k_old) * (gap_mult if gap_days is not None and gw > gap_days else 1.0)
        kl = (k_new if nl < n_new else k_old) * (gap_mult if gap_days is not None and gl > gap_days else 1.0)
        bonus = win_bonus if gap_days is not None and gw > gap_days else 0.0

        new_w = rw + m * kw * surprise + bonus
        new_l = rl - m * kl * surprise
        rating[w], rating[l] = new_w, new_l
        played[w], played[l] = nw + 1, nl + 1
        last[w], last[l] = r.day, r.day

        out.append((r.match_id, r.file_year, r.tourney_start, w, l, rw, rl, new_w, new_l, exp_w))
    res = pd.DataFrame(out, columns=["match_id", "file_year", "tourney_start", "winner_id", "loser_id",
                                     "r_winner", "r_loser", "r_winner_after", "r_loser_after", "exp_w"])
    return res, rating, played

In [ ]:
variants3 = {
    "baseline": {},
    "upset cap 0.7": dict(upset_cap=0.7),
    "upset cap 0.6": dict(upset_cap=0.6),
    "upset cap 0.5": dict(upset_cap=0.5),
    "reset to 1450 after 182 days": dict(reset_days=182, reset_frac=1.0),
    "halfway reset after 182 days": dict(reset_days=182, reset_frac=0.5),
}

rows = []
for name, kw in variants3.items():
    r_, _, _ = run_elo(games, **kw)
    s = score(r_, 2005, 2018)
    rows.append({"variant": name, "accuracy_%": s["accuracy_%"], "log_loss": s["log_loss"], "brier": s["brier"]})

print(pd.DataFrame(rows).sort_values("log_loss").to_string(index=False))

In [ ]:
def snapshot(as_of, min_matches=15, min_recent=0, max_gap_days=182, top=20):
    since = (pd.Timestamp(as_of) - pd.Timedelta(days=364)).strftime("%Y-%m-%d")
    cutoff = (pd.Timestamp(as_of) - pd.Timedelta(days=max_gap_days)).strftime("%Y-%m-%d")
    q = """
    WITH upto AS (
        SELECT * FROM elo_long WHERE tourney_start <= :as_of
    ),
    last_row AS (
        SELECT player_id, MAX(match_id) AS last_match
        FROM upto GROUP BY player_id
    ),
    stats AS (
        SELECT player_id,
               COUNT(*) AS matches_total,
               SUM(CASE WHEN tourney_start > :since THEN 1 ELSE 0 END) AS matches_52w,
               MAX(tourney_start) AS last_start
        FROM upto GROUP BY player_id
    )
    SELECT p.name, ROUND(u.elo_after) AS elo, s.matches_52w, s.matches_total, s.last_start,
           CASE WHEN s.matches_total < 30 THEN 'provisional' ELSE '' END AS note
    FROM last_row l
    JOIN upto u ON u.player_id = l.player_id AND u.match_id = l.last_match
    JOIN stats s ON s.player_id = l.player_id
    JOIN players p ON p.player_id = l.player_id
    WHERE s.matches_total >= :min_matches
      AND s.matches_52w >= :min_recent
      AND s.last_start >= :cutoff
    ORDER BY u.elo_after DESC
    LIMIT :top
    """
    df = pd.read_sql(q, con, params={"as_of": as_of, "since": since, "cutoff": cutoff,
                                     "min_matches": min_matches, "min_recent": min_recent, "top": top})
    df.insert(0, "rank", range(1, len(df) + 1))
    return df

print(snapshot("2024-12-31").to_string(index=False))

In [ ]:
print(pd.read_sql("""
    SELECT DISTINCT file_year, tourney_name, tourney_start
    FROM matches
    WHERE tourney_start >= '2024-12-27' AND tourney_start <= '2025-01-05'
    ORDER BY tourney_start
""", con).to_string(index=False))

In [ ]:
print(snapshot("2024-12-22", min_recent=15).to_string(index=False))
print(snapshot("2024-12-22", min_recent=10).to_string(index=False))

In [ ]:
def period_view(start, end, min_matches=15, min_in_period=1, max_gap_days=182, top=20):
    upto = elo_long[elo_long["tourney_start"] <= end]
    inside = upto[upto["tourney_start"] >= start]
    before = upto[upto["tourney_start"] < start].groupby("player_id")["elo_after"].last()

    g = inside.groupby("player_id").agg(elo_end=("elo_after", "last"),
                                        peak=("elo_after", "max"),
                                        matches_in_period=("match_id", "count"),
                                        last_start=("tourney_start", "max"))
    g["elo_start"] = before.reindex(g.index)
    g["change"] = g["elo_end"] - g["elo_start"]
    g["matches_total"] = upto.groupby("player_id").size().reindex(g.index)

    cutoff = (pd.Timestamp(end) - pd.Timedelta(days=max_gap_days)).strftime("%Y-%m-%d")
    g = g[(g["matches_total"] >= min_matches) & (g["matches_in_period"] >= min_in_period)
          & (g["last_start"] >= cutoff)]
    g = g.sort_values("elo_end", ascending=False).head(top)
    g = g.join(players.set_index("player_id")["name"])
    g.insert(0, "rank", range(1, len(g) + 1))
    return g[["rank", "name", "elo_start", "elo_end", "change", "peak", "matches_in_period"]].round(0)

print(period_view("2024-01-01", "2024-12-22", min_in_period=15).to_string(index=False))
print(period_view("2024-09-01", "2024-09-30", min_in_period=1, top=10).to_string(index=False))
